# 🔮 Year-by-Year Coastal Classification (IMPROVED)

Run the trained model on **each year's satellite data** (2020-2025) to generate unique classified maps.

## 🆕 Recent Improvements:
- ✅ **Percentile-based normalization** - Handles different atmospheric conditions across years
- ✅ **Spectral refinement rules** - Post-processing to improve classification accuracy
- ✅ **Consistent color mapping** - All years use the same color scheme as 2024 reference
- ✅ **Better feature engineering** - Normalized bands before computing indices
- ✅ **Training data optimization** - Year 2025 uses original coastalImage (model training data)

**Note**: The model was trained on 2025 data (coastalImage folder), so 2025 results are the reference standard. Other years use normalized comparison.

## What This Notebook Does:
- ✅ Load trained Random Forest model
- ✅ Loop through years 2020-2025
- ✅ For each year:
  - Load year-specific satellite bands
  - **Normalize bands** for cross-year consistency
  - Compute spectral indices (NDVI, NDWI, etc.)
  - Extract spatial features
  - Generate classification map
  - **Apply spectral refinement** for improved accuracy
  - Export to `../FRONTEND/data/classified_YYYY.png`
  - Calculate area statistics

## Output:
- `../FRONTEND/data/classified_2020.png` through `classified_2025.png`
- `../results/year_stats.json` - Area statistics for all years

---

## 1. Load Model and Setup

In [8]:
import rasterio
import numpy as np
import joblib
import json
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from scipy.ndimage import generic_filter, uniform_filter
from pathlib import Path

# Load trained model
print("📂 Loading trained model...")
model = joblib.load("../models/coastal_classifier_model.pkl")
scaler = joblib.load("../models/feature_scaler.pkl")

with open("../models/model_metadata.json") as f:
    metadata = json.load(f)

print(f"✅ Model loaded: {metadata['model_type']}")
print(f"   Test Accuracy: {metadata['test_accuracy']*100:.2f}%")
print(f"   Expected features: {metadata['n_features']}")
print(f"   Classes: {metadata['classes']}")

# Class definitions - matching the standard classification color scheme
CLASS_INFO = {
    1: {"name": "Seagrass",  "color": "#2ecc71"},  # green
    2: {"name": "Sand",      "color": "#f39c12"},  # orange
    3: {"name": "Seaweed",   "color": "#8e44ad"},  # purple
    4: {"name": "Water",     "color": "#2980b9"},  # blue
    5: {"name": "Landmass",  "color": "#7f8c8d"},  # grey
}

# Output folder for frontend
frontend_data = Path("../FRONTEND/data")
frontend_data.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Output folder: {frontend_data.absolute()}")

📂 Loading trained model...
✅ Model loaded: RandomForestClassifier (RF + Spatial Features)
   Test Accuracy: 93.12%
   Expected features: 21
   Classes: [1, 2, 3, 4, 5]

📁 Output folder: c:\Users\HP LAPTOP 15s\matlabTiff\notebooks\..\FRONTEND\data


## 2. Define Processing Functions

In [9]:
def find_band_file(year_folder, band_name):
    """Find band file in year folder (handles different naming patterns)."""
    # Special case: B08 can also be named B8A in some satellites
    if band_name == "B08":
        patterns = [
            f"*_B08_(Raw).tiff",
            f"*_B08.tiff",
            f"*_B08*.tif",
            f"*_B8A_(Raw).tiff",  # Alternative NIR band name
            f"*_B8A.tiff",
            f"*_B8A*.tif"
        ]
    else:
        patterns = [
            f"*_{band_name}_(Raw).tiff",
            f"*_{band_name}.tiff",
            f"*_{band_name}*.tif"
        ]
    
    for pattern in patterns:
        matches = glob.glob(str(year_folder / pattern))
        if matches:
            return matches[0]
    return None

def majority_filter(values):
    """Return the most common value in a neighborhood window."""
    vals = values[values > 0]  # ignore nodata (0)
    if len(vals) == 0:
        return 0
    counts = np.bincount(vals.astype(int))
    return np.argmax(counts)

def normalize_band(band, percentile_clip=True):
    """Normalize band values to improve cross-year consistency.
    
    Uses percentile-based normalization to handle different atmospheric conditions.
    """
    if percentile_clip:
        # Clip extreme values (reduces atmospheric/cloud effects)
        p2, p98 = np.percentile(band[band > 0], [2, 98])
        band_clipped = np.clip(band, p2, p98)
        # Normalize to [0, 1] range
        band_norm = (band_clipped - p2) / (p98 - p2 + 1e-10)
    else:
        # Simple min-max normalization
        band_norm = (band - band.min()) / (band.max() - band.min() + 1e-10)
    
    return band_norm

def compute_indices(B02, B03, B04, B08):
    """Compute spectral indices: NDVI, NDWI, BAI.
    
    Note: Input bands should be normalized first for better cross-year consistency.
    """
    # NDVI: (NIR - Red) / (NIR + Red)
    NDVI = (B08 - B04) / (B08 + B04 + 1e-10)
    
    # NDWI: (Green - NIR) / (Green + NIR) - water index
    NDWI = (B03 - B08) / (B03 + B08 + 1e-10)
    
    # BAI: 1/((0.1-Red)^2 + (0.06-NIR)^2) - bareness index
    BAI = 1.0 / ((0.1 - B04)**2 + (0.06 - B08)**2 + 1e-10)
    
    return NDVI, NDWI, BAI

def process_year(year):
    """Process one year's satellite data and generate classified map."""
    print(f"\n{'='*60}")
    print(f"📅 Processing Year: {year}")
    print(f"{'='*60}")
    
    # Special case: 2025 uses the preprocessed image (exactly like notebook 04)
    if year == 2025:
        processed_path = Path("../data/processed/processed_image_with_indices.tif")
        if processed_path.exists():
            print(f"   📁 Using preprocessed image (matches notebook 04)")
            
            # Load preprocessed image directly
            with rasterio.open(processed_path) as src:
                image_data = src.read()
                profile = src.profile.copy()
            
            bands, height, width = image_data.shape
            print(f"   📐 Image size: {bands} bands × {height} × {width} pixels")
            
            # Image already has: B02, B03, B04, B08, NDVI, NDWI, BAI (or texture)
            # Extract what we need
            B02 = image_data[0]
            B03 = image_data[1]
            B04 = image_data[2]
            B08 = image_data[3]
            NDVI = image_data[4]
            NDWI = image_data[5]
            BAI = image_data[6]
            
        else:
            print(f"   ⚠️  Preprocessed image not found, using raw bands")
            year_folder = Path("../data/raw/coastalImage")
            # Fall through to normal processing below
            use_preprocessed = False
    else:
        use_preprocessed = False
        year_folder = Path(f"../data/years/{year}")
    
    # For non-2025 years or if preprocessed file not found
    if year != 2025 or not processed_path.exists():
        if not year_folder.exists():
            print(f"   ⚠️  Folder not found: {year_folder}")
            return None
        
        # Find required bands
        print("   🔍 Looking for band files...")
        B02_file = find_band_file(year_folder, "B02")
        B03_file = find_band_file(year_folder, "B03")
        B04_file = find_band_file(year_folder, "B04")
        B08_file = find_band_file(year_folder, "B08")
        
        if not all([B02_file, B03_file, B04_file, B08_file]):
            print(f"   ❌ Missing required bands for {year}")
            print(f"      B02: {B02_file}")
            print(f"      B03: {B03_file}")
            print(f"      B04: {B04_file}")
            print(f"      B08: {B08_file}")
            return None
        
        print(f"   ✅ Found all required bands")
        
        # Load bands
        print("   📂 Loading bands...")
        with rasterio.open(B02_file) as src:
            B02_raw = src.read(1).astype(float)
            profile = src.profile.copy()
        
        B03_raw = rasterio.open(B03_file).read(1).astype(float)
        B04_raw = rasterio.open(B04_file).read(1).astype(float)
        B08_raw = rasterio.open(B08_file).read(1).astype(float)
        
        height, width = B02_raw.shape
        print(f"   📐 Image size: {height} × {width} pixels")
        
        # Normalize bands for cross-year consistency
        print("   📊 Normalizing bands (percentile-based)...")
        B02 = normalize_band(B02_raw, percentile_clip=True)
        B03 = normalize_band(B03_raw, percentile_clip=True)
        B04 = normalize_band(B04_raw, percentile_clip=True)
        B08 = normalize_band(B08_raw, percentile_clip=True)
        
        # Compute spectral indices on normalized bands
        print("   🧮 Computing spectral indices...")
        NDVI, NDWI, BAI = compute_indices(B02, B03, B04, B08)
    
    # Stack all 7 bands
    image_data = np.stack([B02, B03, B04, B08, NDVI, NDWI, BAI], axis=0)
    
    # Identify nodata pixels
    nodata_mask = (
        np.all(image_data == 0, axis=0) |
        np.any(np.isnan(image_data), axis=0)
    )
    print(f"   🔍 Nodata pixels: {np.sum(nodata_mask):,}")
    
    # Compute spatial features (21 total)
    print("   🔧 Computing spatial neighborhood features...")
    neigh_means = np.zeros_like(image_data)
    neigh_stds = np.zeros_like(image_data)
    
    for b in range(7):
        band = image_data[b].astype('float32')
        mean = uniform_filter(band, size=3)
        sqr_mean = uniform_filter(band**2, size=3)
        var = np.maximum(sqr_mean - mean**2, 0)
        neigh_means[b] = mean
        neigh_stds[b] = np.sqrt(var)
    
    # Stack to 21 features and reshape
    spatial_stack = np.concatenate([image_data, neigh_means, neigh_stds], axis=0)
    features = spatial_stack.reshape(21, -1).T
    features = np.nan_to_num(features, nan=0)
    
    print(f"   📊 Feature array shape: {features.shape}")
    
    # Scale and predict
    print("   🔮 Making predictions...")
    features_scaled = scaler.transform(features)
    prediction_flat = model.predict(features_scaled)
    
    # Zero out nodata pixels
    prediction_flat[nodata_mask.flatten()] = 0
    
    # Reshape to map
    prediction_map = prediction_flat.reshape(height, width)
    
    # Apply majority filter to clean noise
    print("   🧹 Applying majority filter...")
    prediction_clean = generic_filter(
        prediction_map.astype(float),
        majority_filter,
        size=3
    ).astype(np.uint8)
    
    # Apply spectral-based refinement to improve accuracy
    print("   ✨ Applying spectral refinement rules...")
    
    # Rule 1: Very high NDVI + low NDWI = likely Landmass (5)
    high_veg_mask = (NDVI > 0.4) & (NDWI < -0.1)
    prediction_clean[high_veg_mask] = 5
    
    # Rule 2: Very low NDVI + very high NDWI = definitely Water (4)
    water_mask = (NDVI < 0.0) & (NDWI > 0.2)
    prediction_clean[water_mask] = 4
    
    # Rule 3: Moderate NDVI + negative NDWI + bright in visible = likely Sand (2)
    sand_mask = (NDVI > 0.0) & (NDVI < 0.2) & (NDWI < 0.0) & (B04 > 0.5)
    prediction_clean[sand_mask] = 2
    
    # Rule 4: Moderate-high NDVI in water areas = likely Seagrass (1) or Seaweed (3)
    # This is trickier - use depth as proxy (darker water = deeper, favors seaweed)
    shallow_veg_mask = (NDVI > 0.1) & (NDVI < 0.35) & (NDWI > -0.1)
    dark_water = B02 < 0.3  # darker areas (likely deeper)
    
    # Seagrass in shallower areas
    prediction_clean[shallow_veg_mask & ~dark_water] = 1
    # Seaweed in deeper/darker areas or with specific spectral signature
    prediction_clean[shallow_veg_mask & dark_water & (B03 > B04)] = 3
    
    print("   ✅ Refinement complete!")
    
    # Calculate statistics
    valid_pixels = prediction_clean[prediction_clean > 0]
    print(f"\n   📈 Class Distribution:")
    stats = {}
    for cls_id in sorted(np.unique(valid_pixels)):
        count = np.sum(prediction_clean == cls_id)
        pct = (count / len(valid_pixels)) * 100
        name = CLASS_INFO.get(int(cls_id), {}).get('name', f'Class {cls_id}')
        print(f"      {name}: {count:,} pixels ({pct:.1f}%)")
        stats[name] = int(count)
    
    # Generate visualization
    print("\n   🎨 Generating classified map...")
    
    # Build colormap
    unique_classes = sorted([c for c in np.unique(prediction_clean) if c > 0])
    max_class = max(CLASS_INFO.keys())
    colors = ['#ffffff']  # nodata = white
    for i in range(1, max_class + 1):
        if i in CLASS_INFO:
            colors.append(CLASS_INFO[i]["color"])
        else:
            colors.append('#ffffff')
    
    cmap = ListedColormap(colors)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 9))
    
    ax.imshow(
        prediction_clean,
        cmap=cmap,
        vmin=0,
        vmax=max_class,
        interpolation='nearest'
    )
    
    # Add legend
    legend_patches = []
    for cls_id in unique_classes:
        if cls_id in CLASS_INFO:
            info = CLASS_INFO[cls_id]
            pixel_count = np.sum(prediction_clean == cls_id)
            total_valid = np.sum(prediction_clean > 0)
            pct = (pixel_count / total_valid) * 100
            label = f"{info['name']} ({pct:.1f}%)"
            patch = mpatches.Patch(color=info['color'], label=label)
            legend_patches.append(patch)
    
    ax.legend(
        handles=legend_patches,
        loc='lower right',
        fontsize=9,
        framealpha=0.9,
        title="Coastal Classes"
    )
    
    ax.set_title(f"Coastal Classification Map — {year}\n(Random Forest Model)",
                 fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    
    # Save to frontend folder with consistent settings
    output_path = frontend_data / f'classified_{year}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close()
    
    print(f"   💾 Saved: {output_path.name}")
    print(f"   ✅ Year {year} complete!")
    
    return stats

## 3. Process All Years

In [10]:
# Process years 2020-2025
years = [2020, 2021, 2022, 2023, 2024, 2025]
all_year_stats = {}

print("\n🚀 Starting year-by-year classification...\n")

for year in years:
    try:
        stats = process_year(year)
        if stats:
            all_year_stats[str(year)] = stats
    except Exception as e:
        print(f"\n   ❌ Error processing {year}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n\n{'='*60}")
print("✅ ALL YEARS PROCESSED!")
print(f"{'='*60}")
print(f"\n📊 Processed {len(all_year_stats)} years: {list(all_year_stats.keys())}")
print(f"📁 Classified maps saved to: {frontend_data.absolute()}")


🚀 Starting year-by-year classification...


📅 Processing Year: 2020
   🔍 Looking for band files...
   ✅ Found all required bands
   📂 Loading bands...
   📐 Image size: 460 × 475 pixels
   📊 Normalizing bands (percentile-based)...
   🧮 Computing spectral indices...
   🔍 Nodata pixels: 0
   🔧 Computing spatial neighborhood features...
   📊 Feature array shape: (218500, 21)
   🔮 Making predictions...
   🧹 Applying majority filter...
   ✨ Applying spectral refinement rules...
   ✅ Refinement complete!

   📈 Class Distribution:
      Seagrass: 821 pixels (0.4%)
      Sand: 16,676 pixels (7.6%)
      Seaweed: 5,673 pixels (2.6%)
      Water: 162,475 pixels (74.4%)
      Landmass: 32,855 pixels (15.0%)

   🎨 Generating classified map...
   💾 Saved: classified_2020.png
   ✅ Year 2020 complete!

📅 Processing Year: 2021
   🔍 Looking for band files...
   ✅ Found all required bands
   📂 Loading bands...
   📐 Image size: 460 × 475 pixels
   📊 Normalizing bands (percentile-based)...
   🧮 Computing 

## 4. Save Statistics (Optional)

In [11]:
# Save year statistics to JSON
results_folder = Path("../results")
results_folder.mkdir(exist_ok=True)

with open(results_folder / 'year_stats.json', 'w') as f:
    json.dump(all_year_stats, f, indent=2)

print(f"💾 Statistics saved to: {results_folder.absolute()}/year_stats.json")

# Display summary
print("\n📋 Summary by Year:")
print("="*60)
for year, stats in all_year_stats.items():
    print(f"\n{year}:")
    for cls_name, count in stats.items():
        print(f"  {cls_name:<12}: {count:>10,} pixels")

💾 Statistics saved to: c:\Users\HP LAPTOP 15s\matlabTiff\notebooks\..\results/year_stats.json

📋 Summary by Year:

2020:
  Seagrass    :        821 pixels
  Sand        :     16,676 pixels
  Seaweed     :      5,673 pixels
  Water       :    162,475 pixels
  Landmass    :     32,855 pixels

2021:
  Seagrass    :      3,165 pixels
  Sand        :     32,004 pixels
  Seaweed     :     12,503 pixels
  Water       :    141,463 pixels
  Landmass    :     29,365 pixels

2022:
  Seagrass    :      3,256 pixels
  Sand        :     13,496 pixels
  Seaweed     :     15,269 pixels
  Water       :    161,635 pixels
  Landmass    :     24,844 pixels

2023:
  Seagrass    :        523 pixels
  Sand        :     17,297 pixels
  Seaweed     :      2,863 pixels
  Water       :    166,391 pixels
  Landmass    :     31,426 pixels

2024:
  Seagrass    :      2,239 pixels
  Sand        :     14,808 pixels
  Seaweed     :      7,436 pixels
  Water       :    166,399 pixels
  Landmass    :     27,618 pixels



## ✅ Done!

**Improvements Made:**
1. **Percentile-based normalization** (2%-98%) clips extreme values caused by clouds, haze, or sensor variations
2. **Spectral refinement rules** correct common mis classifications using domain knowledge:
   - High NDVI + Low NDWI → Landmass
   - Low NDVI + High NDWI → Water
   - Bright visible + Low NDWI → Sand
   - Moderate NDVI in shallow areas → Seagrass vs Seaweed (based on depth/brightness)
3. **Consistent color mapping** across all years matching the 2024 reference

**Next Steps:**
1. **Rerun this notebook** to regenerate all classified maps with improvements
2. Refresh your browser (Ctrl+F5) to see the new classified maps
3. Each year should now show **accurate colors** and **improved classification**
4. Compare the classified maps across years to see coastal changes!

**Note:** If you want even better accuracy, consider:
- Training the model with samples from multiple years
- Adding more ground truth points from years that still have issues
- Fine-tuning the spectral refinement thresholds based on your specific region